In [2]:
# PROGRAM 6: Write a program using image processing techniques to detect traffic signs based on their color
# from an input image.

import cv2
import numpy as np
import matplotlib.pyplot as plt

# 1. Load image (try .jpeg or .jpg)
image = cv2.imread('traffic_signs.jpeg')
if image is None:
    image = cv2.imread('traffic_signs.jpg')

if image is None:
    raise FileNotFoundError('traffic_signs image not found in current directory!')

output = image.copy()
hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

# 2. HSV ranges for common sign / signal colors
red1 = cv2.inRange(hsv, (0, 80, 80), (10, 255, 255))
red2 = cv2.inRange(hsv, (170, 80, 80), (180, 255, 255))
red = red1 | red2

blue = cv2.inRange(hsv, (90, 80, 70), (130, 255, 255))
yellow = cv2.inRange(hsv, (18, 80, 80), (40, 255, 255))

combined = red | blue | yellow

# 3. Morphological operations (remove noise & close gaps)
kernel = np.ones((5, 5), np.uint8)
combined = cv2.morphologyEx(combined, cv2.MORPH_OPEN, kernel)
combined = cv2.morphologyEx(combined, cv2.MORPH_CLOSE, kernel)

# 4. Find contours
contours, _ = cv2.findContours(combined, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# 5. Filter contours based on area and circularity/shape
detected_count = 0
for cnt in contours:
    area = cv2.contourArea(cnt)
    if area < 150:
        continue

    perimeter = cv2.arcLength(cnt, True)
    if perimeter == 0:
        continue

    circularity = 4 * np.pi * area / (perimeter * perimeter)

    if circularity > 0.35:
        x, y, w, h = cv2.boundingRect(cnt)
        detected_count += 1
        cv2.rectangle(output, (x, y), (x + w, y + h), (0, 255, 0), 2)
        cv2.putText(
            output,
            'Traffic Sign',
            (x, max(y - 8, 15)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (0, 255, 255),
            2
        )

print(f'Detected {detected_count} traffic signs/signals.')
cv2.imwrite('traffic_signs_result.jpg', output)

# 6. Display inline using Matplotlib
output_rgb = cv2.cvtColor(output, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(10, 8))
plt.imshow(output_rgb)
plt.title('Color Based Traffic Sign & Signal Detection')
plt.axis('off')
plt.show()
